=============================================================================
MISSOURI VOTER RESOURCE ALLOCATION PROJECT
=============================================================================
Course: CAPS 5576 - Analytics Applications
Team: Group 1
Project: Prescriptive and Predictive Voter Resource Allocation
Focus: Missouri Presidential Elections (2016, 2020, 2024)
=============================================================================

COLLABORATION NOTE:
-------------------
This notebook is initially built in Greg's environment (SNOWBEARAIR_DB).
Once validated, it will be ported to a shared team Snowflake account
where all 4 team members can collaborate.

=============================================================================

FILES REQUIRED (upload as notebook assets before execution):
------------------------------------------------------------

ELECTION DATA (3 files):
- MO 2016 Election Results.csv
- MO 2020 Election Results.csv
- MO 2024 Election Results.csv

CENSUS DATA - 2016 (5 files):
- MO 2016 Census Income.csv
- MO 2016 Census Education.csv
- MO 2016 Census Race.csv
- MO 2016 Census Commute.csv
- MO 2016 Census Sex by Age.csv

CENSUS DATA - 2020 (5 files):
- MO 2020 Census Income.csv
- MO 2020 Census Education.csv
- MO 2020 Census Race.csv
- MO 2020 Census Commute.csv
- MO 2020 Census Sex by Age.csv

CENSUS DATA - 2024 (5 files):
- MO 2024 Census Income.csv
- MO 2024 Census Education.csv
- MO 2024 Census Race.csv
- MO 2024 Census Commute.csv
- MO 2024 Census Sex by Age.csv

POLLING LOCATION DATA (1 file):
- MO 2020 Polling Locations.csv

SHAPEFILE DATA (5 files):
- MO 2020 Precincts.shp
- MO 2020 Precincts.dbf
- MO 2020 Precincts.shx
- MO 2020 Precincts.prj
- MO 2020 Precincts.cpg

TOTAL: 24 files

=============================================================================

NOTEBOOK STRUCTURE:
-------------------
SECTION 1: Documentation & Setup (Cells 1-5)
SECTION 2: Data Ingestion (Cells 6-12)
SECTION 3: Data Cleaning - Election Data (Cells 13-16)
SECTION 4: Data Cleaning - Census Data (Cells 17-22)
SECTION 5: Data Cleaning - Polling & Shapefile (Cells 23-25)
SECTION 6: Write Staging Tables to Snowflake (Cells 26-27)
SECTION 7: SQL Aggregations & JOINs (Cells 28-34)
SECTION 8: Data Quality Validation (Cells 35-37)
SECTION 9: Exploratory Data Analysis (Cells 38-48)
SECTION 10: Geospatial Analysis - David (Cell 49)
SECTION 11: Summary & Next Steps (Cell 50)

=============================================================================

# Missouri Voter Resource Allocation Project

## Project Goal
Analyze historical presidential election results alongside demographic and geographic data to evaluate how polling resources could potentially be allocated more effectively across Missouri precincts.

## Analysis Focus
The analysis focuses on **three presidential election cycles**: 2016, 2020, and 2024.

Presidential election years were chosen because they produce the **highest voter turnout** and the **most consistent statewide participation**. Focusing on presidential election cycles provides a clearer signal for modeling voter demand and analyzing polling resource allocation across precincts.

## Data Architecture
The project integrates several categories of data:
- **Presidential election results** (precinct-level) - 2016, 2020, 2024
- **Census demographic data** (county-level) - ACS 5-Year Estimates aligned to each election
- **Polling location data** - Physical polling places by precinct
- **Precinct boundaries** (optional) - Geographic shapefiles for spatial analysis

## Programming Paradigms
- **Imperative (Python):** Data loading, cleaning, transformations
- **Declarative (SQL):** Joins, aggregations, analytical queries

## Coding Standards
- snake_case naming with meaningful variable names
- One output per cell
- Explanations in Markdown cells (not inline comments)
- Pandas method chaining
- List comprehensions over loops

# Data Sources

## Election Results Data

**Source:** OpenElections Project (GitHub)  
https://github.com/openelections/openelections-data-mo

**Files:**
- MO 2016 Election Results.csv (128,859 rows)
- MO 2020 Election Results.csv (132,123 rows)
- MO 2024 Election Results.csv (190,270 rows)

These datasets contain precinct-level vote totals for each candidate and office. Each row represents the vote total for one candidate within a specific precinct.

**Schema Note:** The 2020 and 2024 files include a `precinct_code` column not present in 2016. This column is dropped during preprocessing to maintain a consistent schema.

## Census Demographic Data

**Source:** U.S. Census Bureau – American Community Survey (ACS) 5-Year Estimates  
https://data.census.gov

Demographic data was aligned with each presidential election year:
- **2016 Election** → ACS 2012-2016
- **2020 Election** → ACS 2016-2020
- **2024 Election** → ACS 2020-2024

**Tables Used (5 per year = 15 files total):**
- B01001 – Sex by Age
- B02001 – Race
- B08301 – Commuting / Transportation to Work
- B15003 – Educational Attainment
- B19013 – Median Household Income

All ACS datasets are at the **county level** for Missouri.

## Polling Location Data

**Source:** MIT Election Data and Science Lab  
https://electionlab.mit.edu/data

**File:** MO 2020 Polling Locations.csv (14,354 rows)

Contains polling locations associated with Missouri precincts. Although the data represents the 2020 election cycle, polling locations generally change slowly, making the dataset suitable for analysis across nearby election years.

## Precinct Geographic Files

**Source:** U.S. Census TIGER/Line Shapefiles  

**Files:** MO 2020 Precincts (.shp, .dbf, .shx, .prj, .cpg)

These files represent Missouri Voting Tabulation District (precinct) boundaries. They may be used for:
- Mapping turnout geographically
- Spatial analysis of polling locations
- Calculating distances between voters and polling places

# Data Preprocessing Requirements

## Election Data Preprocessing

1. **Add election year column** - Enables combining datasets from different cycles
2. **Remove precinct_code column** - Not present in 2016, dropped from 2020/2024
3. **Normalize precinct names** - Uppercase, remove special characters, trim whitespace
4. **Normalize county names** - Uppercase and trim whitespace
5. **Filter to Presidential results only** - Focus analysis on highest-turnout races

## Census Data Preprocessing

1. **Remove header artifact row** - ACS exports include a metadata row where GEO_ID = "Geography"
2. **Remove blank export columns** - Drop any "Unnamed" columns from export artifacts
3. **Extract county name** - Parse from NAME field, removing " County, Missouri" suffix
4. **Add census year column** - Track which ACS vintage the data comes from
5. **Convert to numeric types** - Ensure proper data types for analysis

In [ ]:
%%sql -r dataframe_17
-- Cell: Environment Setup

USE DATABASE SNOWBEARAIR_DB;
CREATE SCHEMA IF NOT EXISTS VOTER_PROJECT;
USE SCHEMA VOTER_PROJECT;
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;
USE ROLE TRAINING_ROLE;

In [ ]:
# Library Imports and Session Initialization

from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import geopandas as gpd

session = get_active_session()

print("=" * 70)
print("ENVIRONMENT INITIALIZED")
print("=" * 70)
print(f"Database:  {session.get_current_database()}")
print(f"Schema:    {session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print("=" * 70)

---
# SECTION 2: Data Ingestion
---

Load all raw data files into pandas DataFrames. Files are loaded exactly as downloaded to preserve raw source data. All transformations occur programmatically to ensure reproducibility.

In [ ]:
# Load Election Results (All Three Years)

raw_election_2016 = pd.read_csv(
    'MO 2016 Election Results.csv', 
    keep_default_na=False, 
    na_values=['']
)

raw_election_2020 = pd.read_csv(
    'MO 2020 Election Results.csv', 
    keep_default_na=False, 
    na_values=[''], 
    low_memory=False
)

raw_election_2024 = pd.read_csv(
    'MO 2024 Election Results.csv', 
    keep_default_na=False, 
    na_values=['']
)

print("=" * 70)
print("ELECTION DATA LOADED")
print("=" * 70)
print(f"2016: {raw_election_2016.shape[0]:>10,} rows × {raw_election_2016.shape[1]} columns")
print(f"2020: {raw_election_2020.shape[0]:>10,} rows × {raw_election_2020.shape[1]} columns")
print(f"2024: {raw_election_2024.shape[0]:>10,} rows × {raw_election_2024.shape[1]} columns")
print("=" * 70)

In [ ]:
# Load Census Data - 2016 (ACS 2012-2016)

raw_census_2016_income = pd.read_csv('MO 2016 Census Income.csv', keep_default_na=False, na_values=[''])
raw_census_2016_education = pd.read_csv('MO 2016 Census Education.csv', keep_default_na=False, na_values=[''])
raw_census_2016_race = pd.read_csv('MO 2016 Census Race.csv', keep_default_na=False, na_values=[''])
raw_census_2016_commute = pd.read_csv('MO 2016 Census Commute.csv', keep_default_na=False, na_values=[''])
raw_census_2016_sex_age = pd.read_csv('MO 2016 Census Sex by Age.csv', keep_default_na=False, na_values=[''])

print("=" * 70)
print("CENSUS DATA LOADED - 2016 (ACS 2012-2016)")
print("=" * 70)
print(f"Income:      {raw_census_2016_income.shape[0]:>4} rows × {raw_census_2016_income.shape[1]:>3} columns")
print(f"Education:   {raw_census_2016_education.shape[0]:>4} rows × {raw_census_2016_education.shape[1]:>3} columns")
print(f"Race:        {raw_census_2016_race.shape[0]:>4} rows × {raw_census_2016_race.shape[1]:>3} columns")
print(f"Commute:     {raw_census_2016_commute.shape[0]:>4} rows × {raw_census_2016_commute.shape[1]:>3} columns")
print(f"Sex by Age:  {raw_census_2016_sex_age.shape[0]:>4} rows × {raw_census_2016_sex_age.shape[1]:>3} columns")
print("=" * 70)

In [ ]:
# Load Census Data - 2020 (ACS 2016-2020)

raw_census_2020_income = pd.read_csv('MO 2020 Census Income.csv', keep_default_na=False, na_values=[''])
raw_census_2020_education = pd.read_csv('MO 2020 Census Education.csv', keep_default_na=False, na_values=[''])
raw_census_2020_race = pd.read_csv('MO 2020 Census Race.csv', keep_default_na=False, na_values=[''])
raw_census_2020_commute = pd.read_csv('MO 2020 Census Commute.csv', keep_default_na=False, na_values=[''])
raw_census_2020_sex_age = pd.read_csv('MO 2020 Census Sex by Age.csv', keep_default_na=False, na_values=[''])

print("=" * 70)
print("CENSUS DATA LOADED - 2020 (ACS 2016-2020)")
print("=" * 70)
print(f"Income:      {raw_census_2020_income.shape[0]:>4} rows × {raw_census_2020_income.shape[1]:>3} columns")
print(f"Education:   {raw_census_2020_education.shape[0]:>4} rows × {raw_census_2020_education.shape[1]:>3} columns")
print(f"Race:        {raw_census_2020_race.shape[0]:>4} rows × {raw_census_2020_race.shape[1]:>3} columns")
print(f"Commute:     {raw_census_2020_commute.shape[0]:>4} rows × {raw_census_2020_commute.shape[1]:>3} columns")
print(f"Sex by Age:  {raw_census_2020_sex_age.shape[0]:>4} rows × {raw_census_2020_sex_age.shape[1]:>3} columns")
print("=" * 70)

In [ ]:
# Load Census Data - 2024 (ACS 2020-2024)

raw_census_2024_income = pd.read_csv('MO 2024 Census Income.csv', keep_default_na=False, na_values=[''])
raw_census_2024_education = pd.read_csv('MO 2024 Census Education.csv', keep_default_na=False, na_values=[''])
raw_census_2024_race = pd.read_csv('MO 2024 Census Race.csv', keep_default_na=False, na_values=[''])
raw_census_2024_commute = pd.read_csv('MO 2024 Census Commute.csv', keep_default_na=False, na_values=[''])
raw_census_2024_sex_age = pd.read_csv('MO 2024 Census Sex by Age.csv', keep_default_na=False, na_values=[''])

print("=" * 70)
print("CENSUS DATA LOADED - 2024 (ACS 2020-2024)")
print("=" * 70)
print(f"Income:      {raw_census_2024_income.shape[0]:>4} rows × {raw_census_2024_income.shape[1]:>3} columns")
print(f"Education:   {raw_census_2024_education.shape[0]:>4} rows × {raw_census_2024_education.shape[1]:>3} columns")
print(f"Race:        {raw_census_2024_race.shape[0]:>4} rows × {raw_census_2024_race.shape[1]:>3} columns")
print(f"Commute:     {raw_census_2024_commute.shape[0]:>4} rows × {raw_census_2024_commute.shape[1]:>3} columns")
print(f"Sex by Age:  {raw_census_2024_sex_age.shape[0]:>4} rows × {raw_census_2024_sex_age.shape[1]:>3} columns")
print("=" * 70)

In [ ]:
# Load Polling Locations Data

raw_polling_locations = pd.read_csv(
    'MO 2020 Polling Locations.csv', 
    keep_default_na=False, 
    na_values=['']
)

print("=" * 70)
print("POLLING LOCATIONS DATA LOADED")
print("=" * 70)
print(f"Rows:    {raw_polling_locations.shape[0]:,}")
print(f"Columns: {raw_polling_locations.shape[1]}")
print(f"Columns: {raw_polling_locations.columns.tolist()}")
print("=" * 70)

In [ ]:
# Load Shapefile Data (Precinct Boundaries)
raw_precincts_gdf = gpd.read_file('MO 2020 Precincts.shp')

print("=" * 70)
print("SHAPEFILE DATA LOADED")
print("=" * 70)
print(f"Precincts: {raw_precincts_gdf.shape[0]:,}")
print(f"Columns:   {raw_precincts_gdf.shape[1]}")
print(f"CRS:       {raw_precincts_gdf.crs}")
print(f"Columns:   {raw_precincts_gdf.columns.tolist()}")
print("=" * 70)

---
# SECTION 3: Data Cleaning - Election Data
---

Clean and standardize election results across all three years:
1. Add year column
2. Drop precinct_code (2020/2024 only)
3. Normalize precinct and county names
4. Filter to Presidential results only
5. Combine into single dataset

In [ ]:
# Define Precinct Name Normalization Function

def normalize_precinct_name(name):
    """
    Standardize precinct names for consistent joining across datasets.
    Missouri precinct names vary (e.g., 'Ward 1', 'WARD 1', 'Ward #1').
    """
    return (
        str(name)
        .upper()
        .replace('#', '')
        .replace('  ', ' ')
        .strip()
    )

print("✅ normalize_precinct_name() function defined")

In [ ]:
# Clean and Filter Election Results (Presidential Only)

election_2016_clean = (
    raw_election_2016
    .query("office == 'President'")
    .assign(year=2016)
    .assign(precinct_clean=lambda x: x['precinct'].apply(normalize_precinct_name))
    .assign(county_clean=lambda x: x['county'].str.upper().str.strip())
    [['year', 'county', 'county_clean', 'precinct', 'precinct_clean', 
      'office', 'district', 'candidate', 'party', 'votes']]
    .reset_index(drop=True)
)

election_2020_clean = (
    raw_election_2020
    .query("office == 'President'")
    .drop(columns=['precinct_code'], errors='ignore')
    .assign(year=2020)
    .assign(precinct_clean=lambda x: x['precinct'].apply(normalize_precinct_name))
    .assign(county_clean=lambda x: x['county'].str.upper().str.strip())
    [['year', 'county', 'county_clean', 'precinct', 'precinct_clean', 
      'office', 'district', 'candidate', 'party', 'votes']]
    .reset_index(drop=True)
)

election_2024_clean = (
    raw_election_2024
    .query("office == 'President'")
    .drop(columns=['precinct_code'], errors='ignore')
    .assign(year=2024)
    .assign(precinct_clean=lambda x: x['precinct'].apply(normalize_precinct_name))
    .assign(county_clean=lambda x: x['county'].str.upper().str.strip())
    [['year', 'county', 'county_clean', 'precinct', 'precinct_clean', 
      'office', 'district', 'candidate', 'party', 'votes']]
    .reset_index(drop=True)
)

print("=" * 70)
print("PRESIDENTIAL ELECTION DATA CLEANED")
print("=" * 70)
print(f"2016 Presidential: {election_2016_clean.shape[0]:>8,} rows")
print(f"2020 Presidential: {election_2020_clean.shape[0]:>8,} rows")
print(f"2024 Presidential: {election_2024_clean.shape[0]:>8,} rows")
print("=" * 70)

In [ ]:
# Combine All Election Years into Single Dataset

all_elections_df = pd.concat(
    [election_2016_clean, election_2020_clean, election_2024_clean], 
    ignore_index=True
)

print("=" * 70)
print("COMBINED ELECTION DATASET")
print("=" * 70)
print(f"Total Rows:    {all_elections_df.shape[0]:,}")
print(f"Total Columns: {all_elections_df.shape[1]}")
print(f"Years:         {sorted(all_elections_df['year'].unique().tolist())}")
print(f"Counties:      {all_elections_df['county_clean'].nunique()}")
print(f"Parties:       {all_elections_df['party'].unique().tolist()}")
print("=" * 70)
all_elections_df.head(10)

---
# SECTION 4: Data Cleaning - Census Data
---

Clean census data for all three years. Each census year is processed identically:
1. Remove header artifact row (GEO_ID = "Geography")
2. Remove state-level totals
3. Extract county FIPS and clean county name
4. Add census year column
5. Rename ACS codes to readable names
6. Convert to numeric types
7. Calculate derived metrics (percentages)

In [ ]:
# Define Census Cleaning Functions

def clean_census_income(df, census_year):
    """Clean census income data for a given year."""
    return (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', '').str.replace(' city', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={'B19013_001E': 'median_household_income'})
        .assign(median_household_income=lambda x: pd.to_numeric(x['median_household_income'], errors='coerce'))
        [['census_year', 'county_fips', 'county_name', 'county_clean', 'median_household_income']]
        .reset_index(drop=True)
    )

def clean_census_education(df, census_year):
    """Clean census education data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', '').str.replace(' city', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={
            'B15003_001E': 'total_pop_25_plus',
            'B15003_017E': 'hs_diploma',
            'B15003_022E': 'bachelors_degree',
            'B15003_023E': 'masters_degree',
            'B15003_024E': 'professional_degree',
            'B15003_025E': 'doctorate_degree'
        })
        .reset_index(drop=True)
    )
    
    for col in ['total_pop_25_plus', 'hs_diploma', 'bachelors_degree', 'masters_degree', 'professional_degree', 'doctorate_degree']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['pct_bachelors_plus'] = (
        (result['bachelors_degree'] + result['masters_degree'] + 
         result['professional_degree'] + result['doctorate_degree']) 
        / result['total_pop_25_plus'] * 100
    ).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_pop_25_plus', 'pct_bachelors_plus']]

def clean_census_race(df, census_year):
    """Clean census race data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', '').str.replace(' city', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={
            'B02001_001E': 'total_population',
            'B02001_002E': 'white_alone',
            'B02001_003E': 'black_alone'
        })
        .reset_index(drop=True)
    )
    
    for col in ['total_population', 'white_alone', 'black_alone']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['pct_white'] = (result['white_alone'] / result['total_population'] * 100).round(2)
    result['pct_minority'] = ((result['total_population'] - result['white_alone']) / result['total_population'] * 100).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_population', 'pct_white', 'pct_minority']]

def clean_census_commute(df, census_year):
    """Clean census commute data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', '').str.replace(' city', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .rename(columns={
            'B08301_001E': 'total_workers',
            'B08301_003E': 'drove_alone',
            'B08301_010E': 'public_transit',
            'B08301_019E': 'walked'
        })
        .reset_index(drop=True)
    )
    
    for col in ['total_workers', 'drove_alone', 'public_transit', 'walked']:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['pct_no_vehicle'] = ((result['public_transit'] + result['walked']) / result['total_workers'] * 100).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_workers', 'pct_no_vehicle']]

def clean_census_sex_age(df, census_year):
    """Clean census sex by age data for a given year."""
    result = (
        df
        .query("GEO_ID != 'Geography'")
        .query("GEO_ID.str.contains('0500000US', na=False)")
        .assign(census_year=census_year)
        .assign(county_fips=lambda x: x['GEO_ID'].str[-5:])
        .assign(county_name=lambda x: x['NAME'].str.replace(', Missouri', '').str.replace(' County', '').str.replace(' city', ''))
        .assign(county_clean=lambda x: x['county_name'].str.upper().str.strip())
        .reset_index(drop=True)
    )
    
    for col in result.columns:
        if col.startswith('B01001'):
            result[col] = pd.to_numeric(result[col], errors='coerce')
    
    result['total_population'] = result['B01001_001E']
    
    male_18plus_cols = [f'B01001_{str(i).zfill(3)}E' for i in range(7, 26)]
    female_18plus_cols = [f'B01001_{str(i).zfill(3)}E' for i in range(31, 50)]
    
    male_18plus_cols = [c for c in male_18plus_cols if c in result.columns]
    female_18plus_cols = [c for c in female_18plus_cols if c in result.columns]
    
    result['voting_age_population'] = result[male_18plus_cols].sum(axis=1) + result[female_18plus_cols].sum(axis=1)
    result['pct_voting_age'] = (result['voting_age_population'] / result['total_population'] * 100).round(2)
    
    return result[['census_year', 'county_fips', 'county_name', 'county_clean', 
                   'total_population', 'voting_age_population', 'pct_voting_age']]

print("✅ Census cleaning functions defined")

In [ ]:
# Clean Census Income Data (All Years)

census_income_2016 = clean_census_income(raw_census_2016_income, 2016)
census_income_2020 = clean_census_income(raw_census_2020_income, 2020)
census_income_2024 = clean_census_income(raw_census_2024_income, 2024)

all_census_income_df = pd.concat(
    [census_income_2016, census_income_2020, census_income_2024],
    ignore_index=True
)

print(f"✅ Census Income cleaned: {all_census_income_df.shape[0]} rows ({all_census_income_df['census_year'].nunique()} years)")
all_census_income_df.head()

In [ ]:
# Clean Census Education Data (All Years)

census_education_2016 = clean_census_education(raw_census_2016_education, 2016)
census_education_2020 = clean_census_education(raw_census_2020_education, 2020)
census_education_2024 = clean_census_education(raw_census_2024_education, 2024)

all_census_education_df = pd.concat(
    [census_education_2016, census_education_2020, census_education_2024],
    ignore_index=True
)

print(f"✅ Census Education cleaned: {all_census_education_df.shape[0]} rows ({all_census_education_df['census_year'].nunique()} years)")
all_census_education_df.head()

In [ ]:
# Clean Census Race Data (All Years)

census_race_2016 = clean_census_race(raw_census_2016_race, 2016)
census_race_2020 = clean_census_race(raw_census_2020_race, 2020)
census_race_2024 = clean_census_race(raw_census_2024_race, 2024)

all_census_race_df = pd.concat(
    [census_race_2016, census_race_2020, census_race_2024],
    ignore_index=True
)

print(f"✅ Census Race cleaned: {all_census_race_df.shape[0]} rows ({all_census_race_df['census_year'].nunique()} years)")
all_census_race_df.head()

In [ ]:
# Clean Census Commute Data (All Years)

census_commute_2016 = clean_census_commute(raw_census_2016_commute, 2016)
census_commute_2020 = clean_census_commute(raw_census_2020_commute, 2020)
census_commute_2024 = clean_census_commute(raw_census_2024_commute, 2024)

all_census_commute_df = pd.concat(
    [census_commute_2016, census_commute_2020, census_commute_2024],
    ignore_index=True
)

print(f"✅ Census Commute cleaned: {all_census_commute_df.shape[0]} rows ({all_census_commute_df['census_year'].nunique()} years)")
all_census_commute_df.head()

In [ ]:
# Clean Census Sex by Age Data (All Years)

census_sex_age_2016 = clean_census_sex_age(raw_census_2016_sex_age, 2016)
census_sex_age_2020 = clean_census_sex_age(raw_census_2020_sex_age, 2020)
census_sex_age_2024 = clean_census_sex_age(raw_census_2024_sex_age, 2024)

all_census_sex_age_df = pd.concat(
    [census_sex_age_2016, census_sex_age_2020, census_sex_age_2024],
    ignore_index=True
)

print(f"✅ Census Sex/Age cleaned: {all_census_sex_age_df.shape[0]} rows ({all_census_sex_age_df['census_year'].nunique()} years)")
all_census_sex_age_df.head()

---
# SECTION 5: Data Cleaning - Polling Locations & Shapefile
---

Clean polling locations and prepare shapefile for potential geospatial analysis.

In [ ]:
# Clean Polling Locations Data

polling_locations_df = (
    raw_polling_locations
    .copy()
    .assign(county_clean=lambda x: x['county_name'].str.strip().str.upper())
    .assign(precinct_clean=lambda x: x['precinct_name'].apply(normalize_precinct_name))
    .assign(polling_address=lambda x: x['address'].str.strip())
)

print("=" * 70)
print("POLLING LOCATIONS CLEANED")
print("=" * 70)
print(f"Total Rows:           {polling_locations_df.shape[0]:,}")
print(f"Unique Counties:      {polling_locations_df['county_clean'].nunique()}")
print(f"Unique Precincts:     {polling_locations_df['precinct_clean'].nunique()}")
print(f"Unique Polling Places: {polling_locations_df['polling_place_id'].nunique()}")
print("=" * 70)
polling_locations_df.head()

In [ ]:
# Prepare Shapefile Data

precincts_df = (
    raw_precincts_gdf
    .drop(columns=['geometry'])
    .assign(county_fips=lambda x: '29' + x['COUNTYFP20'])
    .assign(precinct_name=lambda x: x['NAME20'])
    .assign(precinct_clean=lambda x: x['NAME20'].apply(normalize_precinct_name))
    .assign(land_area_sqm=lambda x: x['ALAND20'])
    .assign(water_area_sqm=lambda x: x['AWATER20'])
    .assign(centroid_lat=lambda x: pd.to_numeric(x['INTPTLAT20'], errors='coerce'))
    .assign(centroid_lon=lambda x: pd.to_numeric(x['INTPTLON20'], errors='coerce'))
    [['county_fips', 'COUNTYFP20', 'precinct_name', 'precinct_clean', 
      'GEOID20', 'land_area_sqm', 'water_area_sqm', 'centroid_lat', 'centroid_lon']]
    .rename(columns={'COUNTYFP20': 'county_fips_3', 'GEOID20': 'geoid'})
)

print("=" * 70)
print("SHAPEFILE DATA PREPARED (Tabular - No Geometry)")
print("=" * 70)
print(f"Precincts: {precincts_df.shape[0]:,}")
print(f"Columns:   {precincts_df.columns.tolist()}")
print("=" * 70)
precincts_df.head()

---
# SECTION 6: Write Staging Tables to Snowflake
---

Write all cleaned DataFrames to Snowflake staging tables. These tables will be used for SQL-based aggregations and joins.

In [ ]:
# Write All Staging Tables to Snowflake

staging_tables = {
    'STG_ELECTION_RESULTS': all_elections_df,
    'STG_CENSUS_INCOME': all_census_income_df,
    'STG_CENSUS_EDUCATION': all_census_education_df,
    'STG_CENSUS_RACE': all_census_race_df,
    'STG_CENSUS_COMMUTE': all_census_commute_df,
    'STG_CENSUS_SEX_AGE': all_census_sex_age_df,
    'STG_POLLING_LOCATIONS': polling_locations_df,
    'STG_PRECINCTS': precincts_df
}

print("=" * 70)
print("WRITING STAGING TABLES TO SNOWFLAKE")
print("=" * 70)

for table_name, df in staging_tables.items():
    session.write_pandas(df, table_name, auto_create_table=True, overwrite=True)
    print(f"✅ {table_name}: {len(df):,} rows")

print("=" * 70)
print("All staging tables written successfully!")
print("=" * 70)

---
# SECTION 7: SQL Aggregations & JOINs (Declarative)
---

Create analytical tables using SQL:
- Aggregate precinct-level votes to county level
- Pivot turnout by year
- Join election results with census demographics

In [ ]:
%%sql -r dataframe_1
-- Cell: Create Precinct Turnout Summary

-- Aggregate votes to precinct level for each year
-- Calculate party vote shares

CREATE OR REPLACE TABLE PRECINCT_TURNOUT AS
SELECT 
    year,
    county_clean AS county,
    precinct_clean AS precinct,
    SUM(votes) AS total_votes,
    SUM(CASE WHEN party = 'REP' THEN votes ELSE 0 END) AS republican_votes,
    SUM(CASE WHEN party = 'DEM' THEN votes ELSE 0 END) AS democrat_votes,
    SUM(CASE WHEN party NOT IN ('REP', 'DEM') OR party IS NULL THEN votes ELSE 0 END) AS other_votes,
    ROUND(SUM(CASE WHEN party = 'REP' THEN votes ELSE 0 END) / NULLIF(SUM(votes), 0) * 100, 2) AS republican_pct,
    ROUND(SUM(CASE WHEN party = 'DEM' THEN votes ELSE 0 END) / NULLIF(SUM(votes), 0) * 100, 2) AS democrat_pct
FROM STG_ELECTION_RESULTS
GROUP BY year, county_clean, precinct_clean
ORDER BY year, county, precinct;

SELECT 'PRECINCT_TURNOUT' AS table_name, COUNT(*) AS row_count FROM PRECINCT_TURNOUT;

In [ ]:
%%sql -r dataframe_2
-- Cell: Create County Turnout by Year

-- Aggregate to county level for each year

CREATE OR REPLACE TABLE COUNTY_TURNOUT AS
SELECT 
    year,
    county,
    COUNT(DISTINCT precinct) AS precinct_count,
    SUM(total_votes) AS total_votes,
    SUM(republican_votes) AS republican_votes,
    SUM(democrat_votes) AS democrat_votes,
    SUM(other_votes) AS other_votes,
    ROUND(SUM(republican_votes) / NULLIF(SUM(total_votes), 0) * 100, 2) AS republican_pct,
    ROUND(SUM(democrat_votes) / NULLIF(SUM(total_votes), 0) * 100, 2) AS democrat_pct
FROM PRECINCT_TURNOUT
GROUP BY year, county
ORDER BY year, county;

SELECT 'COUNTY_TURNOUT' AS table_name, COUNT(*) AS row_count FROM COUNTY_TURNOUT;

In [ ]:
%%sql -r dataframe_3
-- Cell: Create County Turnout Trend (Pivot)

-- Pivot to show all years side by side for trend analysis

CREATE OR REPLACE TABLE COUNTY_TURNOUT_TREND AS
SELECT 
    county,
    MAX(CASE WHEN year = 2016 THEN precinct_count END) AS precincts_2016,
    MAX(CASE WHEN year = 2020 THEN precinct_count END) AS precincts_2020,
    MAX(CASE WHEN year = 2024 THEN precinct_count END) AS precincts_2024,
    MAX(CASE WHEN year = 2016 THEN total_votes END) AS votes_2016,
    MAX(CASE WHEN year = 2020 THEN total_votes END) AS votes_2020,
    MAX(CASE WHEN year = 2024 THEN total_votes END) AS votes_2024,
    MAX(CASE WHEN year = 2016 THEN republican_pct END) AS rep_pct_2016,
    MAX(CASE WHEN year = 2020 THEN republican_pct END) AS rep_pct_2020,
    MAX(CASE WHEN year = 2024 THEN republican_pct END) AS rep_pct_2024,
    MAX(CASE WHEN year = 2016 THEN democrat_pct END) AS dem_pct_2016,
    MAX(CASE WHEN year = 2020 THEN democrat_pct END) AS dem_pct_2020,
    MAX(CASE WHEN year = 2024 THEN democrat_pct END) AS dem_pct_2024
FROM COUNTY_TURNOUT
GROUP BY county
ORDER BY county;

SELECT 'COUNTY_TURNOUT_TREND' AS table_name, COUNT(*) AS row_count FROM COUNTY_TURNOUT_TREND;

In [ ]:
%%sql -r dataframe_4
-- Cell: Create Census Demographics Combined (Pivot by Year)

-- Combine all census tables and pivot to get one row per county with all years

CREATE OR REPLACE TABLE COUNTY_CENSUS AS
SELECT 
    i.county_clean AS county,
    i.county_fips,
    i.county_name,
    i.census_year,
    i.median_household_income,
    e.pct_bachelors_plus,
    r.total_population,
    r.pct_minority,
    c.pct_no_vehicle,
    s.voting_age_population,
    s.pct_voting_age
FROM STG_CENSUS_INCOME i
LEFT JOIN STG_CENSUS_EDUCATION e 
    ON i.county_clean = e.county_clean AND i.census_year = e.census_year
LEFT JOIN STG_CENSUS_RACE r 
    ON i.county_clean = r.county_clean AND i.census_year = r.census_year
LEFT JOIN STG_CENSUS_COMMUTE c 
    ON i.county_clean = c.county_clean AND i.census_year = c.census_year
LEFT JOIN STG_CENSUS_SEX_AGE s 
    ON i.county_clean = s.county_clean AND i.census_year = s.census_year
ORDER BY i.county_clean, i.census_year;

SELECT 'COUNTY_CENSUS' AS table_name, COUNT(*) AS row_count FROM COUNTY_CENSUS;

In [ ]:
%%sql -r dataframe_5
-- Cell: Create Master County Analysis Table

-- Join turnout trends with census data for comprehensive analysis table

CREATE OR REPLACE TABLE COUNTY_ANALYSIS AS
SELECT 
    t.county,
    
    -- Turnout by year
    t.votes_2016,
    t.votes_2020,
    t.votes_2024,
    t.rep_pct_2016,
    t.rep_pct_2020,
    t.rep_pct_2024,
    t.dem_pct_2016,
    t.dem_pct_2020,
    t.dem_pct_2024,
    
    -- 2016 Census
    c16.total_population AS pop_2016,
    c16.voting_age_population AS vap_2016,
    c16.median_household_income AS income_2016,
    c16.pct_bachelors_plus AS edu_2016,
    c16.pct_minority AS minority_2016,
    
    -- 2020 Census
    c20.total_population AS pop_2020,
    c20.voting_age_population AS vap_2020,
    c20.median_household_income AS income_2020,
    c20.pct_bachelors_plus AS edu_2020,
    c20.pct_minority AS minority_2020,
    c20.pct_no_vehicle AS no_vehicle_2020,
    
    -- 2024 Census
    c24.total_population AS pop_2024,
    c24.voting_age_population AS vap_2024,
    c24.median_household_income AS income_2024,
    c24.pct_bachelors_plus AS edu_2024,
    c24.pct_minority AS minority_2024,
    
    -- Calculated turnout rates
    ROUND(t.votes_2016 / NULLIF(c16.voting_age_population, 0) * 100, 2) AS turnout_pct_2016,
    ROUND(t.votes_2020 / NULLIF(c20.voting_age_population, 0) * 100, 2) AS turnout_pct_2020,
    ROUND(t.votes_2024 / NULLIF(c24.voting_age_population, 0) * 100, 2) AS turnout_pct_2024

FROM COUNTY_TURNOUT_TREND t
LEFT JOIN COUNTY_CENSUS c16 ON t.county = c16.county AND c16.census_year = 2016
LEFT JOIN COUNTY_CENSUS c20 ON t.county = c20.county AND c20.census_year = 2020
LEFT JOIN COUNTY_CENSUS c24 ON t.county = c24.county AND c24.census_year = 2024
ORDER BY t.county;

SELECT 'COUNTY_ANALYSIS' AS table_name, COUNT(*) AS row_count FROM COUNTY_ANALYSIS;

In [ ]:
%%sql -r dataframe_6
-- Cell: Create Polling Location Summary

-- Aggregate polling locations by county

CREATE OR REPLACE TABLE COUNTY_POLLING_SUMMARY AS
SELECT 
    county_clean AS county,
    COUNT(DISTINCT polling_place_id) AS unique_polling_places,
    COUNT(DISTINCT precinct_clean) AS unique_precincts,
    COUNT(*) AS total_records,
    ROUND(COUNT(DISTINCT precinct_clean) / NULLIF(COUNT(DISTINCT polling_place_id), 0), 2) AS precincts_per_polling_place
FROM STG_POLLING_LOCATIONS
GROUP BY county_clean
ORDER BY precincts_per_polling_place DESC;

SELECT 'COUNTY_POLLING_SUMMARY' AS table_name, COUNT(*) AS row_count FROM COUNTY_POLLING_SUMMARY;

---
# SECTION 8: Data Quality Validation
---

Verify data integrity before proceeding with EDA:
- Row counts for all tables
- County matching validation
- Missing value checks

In [ ]:
# Data Quality Summary - Table Row Counts

print("=" * 70)
print("DATA QUALITY SUMMARY - TABLE ROW COUNTS")
print("=" * 70)

staging_tables = [
    'STG_ELECTION_RESULTS', 'STG_CENSUS_INCOME', 'STG_CENSUS_EDUCATION',
    'STG_CENSUS_RACE', 'STG_CENSUS_COMMUTE', 'STG_CENSUS_SEX_AGE',
    'STG_POLLING_LOCATIONS', 'STG_PRECINCTS'
]

analytical_tables = [
    'PRECINCT_TURNOUT', 'COUNTY_TURNOUT', 'COUNTY_TURNOUT_TREND',
    'COUNTY_CENSUS', 'COUNTY_ANALYSIS', 'COUNTY_POLLING_SUMMARY'
]

print("\n📦 STAGING TABLES:")
for table in staging_tables:
    count = session.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
    print(f"   {table}: {count:,} rows")

print("\n📊 ANALYTICAL TABLES:")
for table in analytical_tables:
    count = session.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
    print(f"   {table}: {count:,} rows")

print("=" * 70)

In [ ]:
%%sql -r dataframe_7
-- Cell: Validate County Matching

-- Check for county mismatches between election and census data

SELECT 
    'Election counties not in Census' AS validation_check,
    COUNT(DISTINCT t.county) AS count
FROM COUNTY_TURNOUT t
LEFT JOIN COUNTY_CENSUS c ON t.county = c.county
WHERE c.county IS NULL

UNION ALL

SELECT 
    'Census counties not in Election' AS validation_check,
    COUNT(DISTINCT c.county) AS count
FROM COUNTY_CENSUS c
LEFT JOIN COUNTY_TURNOUT t ON c.county = t.county
WHERE t.county IS NULL;

---
# SECTION 9: Exploratory Data Analysis (EDA)
---

Analyze patterns in:
- Statewide turnout trends
- County-level turnout variations
- Demographic correlations
- Polling resource distribution

In [ ]:
%%sql -r dataframe_8
-- Cell: Statewide Turnout Summary by Year

-- Overall Missouri turnout by presidential election year

SELECT 
    year,
    COUNT(DISTINCT county) AS counties,
    COUNT(DISTINCT precinct) AS precincts,
    SUM(total_votes) AS total_votes,
    ROUND(SUM(republican_votes) / SUM(total_votes) * 100, 2) AS statewide_rep_pct,
    ROUND(SUM(democrat_votes) / SUM(total_votes) * 100, 2) AS statewide_dem_pct
FROM PRECINCT_TURNOUT
GROUP BY year
ORDER BY year;

In [ ]:
%%sql -r dataframe_9
-- Cell: County Turnout Descriptive Statistics (2020)

-- Descriptive statistics for county-level turnout

SELECT 
    '2020 County Stats' AS metric,
    COUNT(*) AS n_counties,
    ROUND(AVG(total_votes), 0) AS mean_votes,
    ROUND(MEDIAN(total_votes), 0) AS median_votes,
    MIN(total_votes) AS min_votes,
    MAX(total_votes) AS max_votes,
    ROUND(STDDEV(total_votes), 0) AS std_votes
FROM COUNTY_TURNOUT
WHERE year = 2020;

In [ ]:
%%sql -r dataframe_10
-- Cell: Top 10 Counties by Total Votes (2020)

-- Highest vote totals (population centers)

SELECT 
    county,
    total_votes,
    republican_pct,
    democrat_pct,
    precinct_count
FROM COUNTY_TURNOUT
WHERE year = 2020
ORDER BY total_votes DESC
LIMIT 10;

In [ ]:
%%sql -r dataframe_11
-- Cell: Top 10 Counties by Turnout Rate (2020)

-- Highest turnout as percentage of voting age population

SELECT 
    county,
    turnout_pct_2020,
    votes_2020,
    vap_2020 AS voting_age_pop,
    pop_2020 AS total_pop
FROM COUNTY_ANALYSIS
WHERE turnout_pct_2020 IS NOT NULL
ORDER BY turnout_pct_2020 DESC
LIMIT 10;

In [ ]:
%%sql -r dataframe_12
-- Cell: Bottom 10 Counties by Turnout Rate (2020)

-- Lowest turnout - potential areas for resource focus

SELECT 
    county,
    turnout_pct_2020,
    votes_2020,
    vap_2020 AS voting_age_pop,
    pop_2020 AS total_pop
FROM COUNTY_ANALYSIS
WHERE turnout_pct_2020 IS NOT NULL
ORDER BY turnout_pct_2020 ASC
LIMIT 10;

In [ ]:
%%sql -r dataframe_13
-- Cell: Turnout Change 2016 to 2024

-- Counties with largest turnout changes over time

SELECT 
    county,
    votes_2016,
    votes_2020,
    votes_2024,
    votes_2024 - votes_2016 AS vote_change,
    ROUND((votes_2024 - votes_2016) / NULLIF(votes_2016, 0) * 100, 2) AS pct_change
FROM COUNTY_TURNOUT_TREND
WHERE votes_2016 IS NOT NULL AND votes_2024 IS NOT NULL
ORDER BY pct_change DESC
LIMIT 10;

In [ ]:
%%sql -r dataframe_14
-- Cell: Polling Resource Strain Analysis

-- Counties with high precincts-per-polling-place (potential under-resourcing)

SELECT 
    p.county,
    p.unique_polling_places,
    p.unique_precincts,
    p.precincts_per_polling_place,
    a.pop_2020,
    a.votes_2020,
    a.turnout_pct_2020
FROM COUNTY_POLLING_SUMMARY p
LEFT JOIN COUNTY_ANALYSIS a ON p.county = a.county
WHERE p.precincts_per_polling_place > 1
ORDER BY p.precincts_per_polling_place DESC
LIMIT 15;

In [ ]:
%%sql -r dataframe_15
-- Cell: Demographics vs Turnout Snapshot

-- Key demographic indicators alongside turnout

SELECT 
    county,
    pop_2020,
    income_2020,
    edu_2020 AS pct_bachelors,
    minority_2020 AS pct_minority,
    no_vehicle_2020 AS pct_no_vehicle,
    turnout_pct_2020,
    rep_pct_2020,
    dem_pct_2020
FROM COUNTY_ANALYSIS
WHERE turnout_pct_2020 IS NOT NULL
ORDER BY turnout_pct_2020 DESC
LIMIT 20;

In [ ]:
# Correlation Analysis - Demographics vs Turnout

county_analysis_df = session.table('COUNTY_ANALYSIS').to_pandas()

print("=" * 70)
print("CORRELATION MATRIX - DEMOGRAPHICS VS TURNOUT (2020)")
print("=" * 70)

correlation_cols = [
    'INCOME_2020', 'EDU_2020', 'MINORITY_2020', 'NO_VEHICLE_2020',
    'TURNOUT_PCT_2020', 'REP_PCT_2020', 'DEM_PCT_2020'
]

valid_cols = [c for c in correlation_cols if c in county_analysis_df.columns]
correlation_matrix = county_analysis_df[valid_cols].corr().round(3)
print(correlation_matrix)
print("=" * 70)

In [ ]:
# Key Findings Summary

print("=" * 70)
print("KEY FINDINGS SUMMARY")
print("=" * 70)

print("""
1. STATEWIDE TRENDS:
   - Compare total turnout across 2016, 2020, 2024
   - Analyze party vote share changes over time
   
2. TURNOUT PATTERNS:
   - Identify high and low turnout counties
   - Examine turnout rate vs raw vote counts
   - Track changes in turnout over time

3. DEMOGRAPHIC CORRELATIONS:
   - Income vs turnout relationship
   - Education level impact on participation
   - Minority population voting patterns
   - Transportation access and voting

4. POLLING RESOURCES:
   - Counties with high precincts-per-polling-place
   - Cross-reference with population and turnout
   - Identify potential under-resourced areas

5. NEXT STEPS:
   - Predictive modeling for voter demand
   - Prescriptive recommendations for resource allocation
   - Geospatial visualization (David's section)
   - Final presentation preparation
""")
print("=" * 70)

---
# SECTION 10: Geospatial Analysis
---

**Owner: David**

This section is reserved for geospatial analysis using the precinct boundary shapefiles. Potential analyses include:
- Mapping voter turnout by precinct
- Visualizing polling location distribution
- Calculating distances between population centers and polling places
- Identifying geographic clusters of under-resourced areas

In [ ]:
# Geospatial Analysis - David (STUB)

# =============================================================================
# GEOSPATIAL ANALYSIS - DAVID
# =============================================================================
#
# This cell is a starting point for geospatial analysis.
# The shapefile data has been loaded and the tabular version is in STG_PRECINCTS.
#
# AVAILABLE DATA:
# ---------------
# - raw_precincts_gdf: GeoDataFrame with full geometry (loaded in Cell 12)
# - STG_PRECINCTS: Tabular precinct data with centroids (lat/lon)
#
# POTENTIAL ANALYSES:
# -------------------
# 1. Choropleth map of turnout by precinct
# 2. Polling location accessibility analysis
# 3. Distance calculations (voters to nearest polling place)
# 4. Geographic clustering of low-turnout areas
#
# SUGGESTED LIBRARIES:
# --------------------
# - geopandas (already imported)
# - folium (for interactive maps)
# - shapely (for geometry operations)
#
# =============================================================================

# David - add your geospatial analysis code below:

print("=" * 70)
print("GEOSPATIAL ANALYSIS - DAVID")
print("=" * 70)
print("Shapefile loaded: MO 2020 Precincts")
print(f"Total precincts: {raw_precincts_gdf.shape[0]:,}")
print(f"CRS: {raw_precincts_gdf.crs}")
print(f"Geometry types: {raw_precincts_gdf.geom_type.unique().tolist()}")
print("=" * 70)

# Example: View first few precincts
raw_precincts_gdf[['NAME20', 'COUNTYFP20', 'ALAND20', 'INTPTLAT20', 'INTPTLON20']].head()

In [ ]:
%%sql -r dataframe_16
-- Cell: Create Visualization Export Table

-- Final export table for visualization tools (Tableau, Flourish, etc.)

CREATE OR REPLACE TABLE COUNTY_VIZ_EXPORT AS
SELECT 
    a.county,
    a.pop_2016, a.pop_2020, a.pop_2024,
    a.vap_2016, a.vap_2020, a.vap_2024,
    a.votes_2016, a.votes_2020, a.votes_2024,
    a.turnout_pct_2016, a.turnout_pct_2020, a.turnout_pct_2024,
    a.rep_pct_2016, a.rep_pct_2020, a.rep_pct_2024,
    a.dem_pct_2016, a.dem_pct_2020, a.dem_pct_2024,
    a.income_2016, a.income_2020, a.income_2024,
    a.edu_2016, a.edu_2020, a.edu_2024,
    a.minority_2016, a.minority_2020, a.minority_2024,
    a.no_vehicle_2020,
    p.unique_polling_places,
    p.unique_precincts,
    p.precincts_per_polling_place
FROM COUNTY_ANALYSIS a
LEFT JOIN COUNTY_POLLING_SUMMARY p ON a.county = p.county
ORDER BY a.county;

SELECT 'COUNTY_VIZ_EXPORT created' AS status, COUNT(*) AS row_count FROM COUNTY_VIZ_EXPORT;

---
# SECTION 11: Summary & Next Steps
---

## Data Pipeline Complete

### Staging Tables Created:
- `STG_ELECTION_RESULTS` - Combined presidential results (2016, 2020, 2024)
- `STG_CENSUS_INCOME` - Median household income (3 years)
- `STG_CENSUS_EDUCATION` - Educational attainment (3 years)
- `STG_CENSUS_RACE` - Race demographics (3 years)
- `STG_CENSUS_COMMUTE` - Transportation/commute patterns (3 years)
- `STG_CENSUS_SEX_AGE` - Age/sex distribution with VAP (3 years)
- `STG_POLLING_LOCATIONS` - Polling place locations (2020)
- `STG_PRECINCTS` - Precinct boundary data (2020)

### Analytical Tables Created:
- `PRECINCT_TURNOUT` - Precinct-level turnout by year
- `COUNTY_TURNOUT` - County-level turnout by year
- `COUNTY_TURNOUT_TREND` - Turnout pivoted across years
- `COUNTY_CENSUS` - Census demographics by year
- `COUNTY_ANALYSIS` - Master analysis table with all metrics
- `COUNTY_POLLING_SUMMARY` - Polling resource distribution
- `COUNTY_VIZ_EXPORT` - Export-ready for visualization tools

## Next Steps for Team:
1. **Predictive Modeling** - Build models to predict voter demand
2. **Prescriptive Analytics** - Recommend polling resource allocation
3. **Geospatial Analysis** - David's precinct mapping work
4. **Visualization** - Tableau/Flourish dashboards
5. **Final Presentation** - Synthesize findings

In [ ]:
# Final Status

print("=" * 70)
print("✅ NOTEBOOK EXECUTION COMPLETE")
print("=" * 70)
print("""
DATA LOADED:
  • 3 Election years (2016, 2020, 2024) - Presidential results only
  • 15 Census files (5 tables × 3 years)
  • 1 Polling locations file (2020)
  • 1 Shapefile set (precinct boundaries)

TABLES CREATED: 14 total (8 staging + 6 analytical)

READY FOR:
  • Predictive modeling
  • Prescriptive analytics
  • Geospatial analysis (David)
  • Visualization exports
  
COLLABORATION NOTE:
  This notebook will be ported to a shared team Snowflake account
  for full team collaboration.
""")
print("=" * 70)